# Rappel de l objectif rendefinssant les euristic , les meta heur, diff entre diversification et intensification.

## Data set cf. GitHub : https://github.com/PyVRP/VRPLIB

In [149]:
import vrplib

# Remplacez par le vrai nom d'un de vos fichiers (ex: data/C101.txt)
instance = vrplib.read_instance("data/C101.txt", instance_format="solomon") 
solution_opt = vrplib.read_solution("data/C101.sol")

# Afficher ce que contient l'instance
print(instance.keys())
print("Score optimal à battre :", solution_opt.get("cost", solution_opt.get("Cost")))

dict_keys(['name', 'vehicles', 'capacity', 'node_coord', 'demand', 'time_window', 'service_time', 'edge_weight'])
Score optimal à battre : 827.3


## Construction du Graph 


In [150]:
import random
from collections import deque

# =======================================================
# 0. STRUCTURE DE DONNÉES (Classes)
# =======================================================

class Node:
    """Modélise un sommet du réseau (Dépôt ou Client) avec ses paramètres."""
    def __init__(self, node_id, demand=0, earliest_time=0, latest_time=float('inf'), service_time=0):
        self.node_id = node_id  # Indice i
        self.q = demand         # Demande q_i
        self.e = earliest_time  # Ouverture de la fenêtre e_i
        self.l = latest_time    # Fermeture de la fenêtre l_i
        self.s = service_time   # Temps de service s_i

class CVRPTW_Network:
    """Modélise le graphe orienté complet G = (V, E) et les paramètres du CVRPTW."""
    def __init__(self, n_customers, k_trucks, capacity):
        self.N = n_customers    # Nombre total de clients N
        self.K = k_trucks       # Nombre de camions K
        self.Q = capacity       # Capacité maximale d'un camion Q
        self.B = float('inf')   # Borne de coût B'

        self.V = list(range(self.N + 1))
        self.C = [node for node in self.V if node != 0]
        self.E = [(i, j) for i in self.V for j in self.V if i != j]

        self.nodes = {}
        self.travel_times = {}

    def add_node(self, node_id, demand=0, e=0, l=float('inf'), s=0):
        self.nodes[node_id] = Node(node_id, demand, e, l, s)

    def set_travel_time(self, i, j, time_val):
        if (i, j) in self.E:
            self.travel_times[(i, j)] = time_val

    def calculate_objective(self, routes):
        """Calcule la fonction objectif Z = Somme(C_ij * X_ij)."""
        total_time = 0
        for route in routes:
            for idx in range(len(route) - 1):
                i = route[idx]
                j = route[idx + 1]
                total_time += self.travel_times.get((i, j), 0)
        return total_time




In [151]:
def charger_instance_cvrptw(chemin_fichier):
    import vrplib
    # Lecture des données brutes
    data = vrplib.read_instance(chemin_fichier, instance_format="solomon")
    
    # CORRECTION : On compte le nombre de nœuds en regardant la taille de la liste 'demand'
    dimension = len(data['demand'])
    n_clients = dimension - 1 
    capacite = data['capacity']
    nb_camions = data.get('vehicles', 25) # Par défaut, Solomon utilise souvent 25 véhicules
    
    # Création de votre objet réseau
    reseau = CVRPTW_Network(n_customers=n_clients, k_trucks=nb_camions, capacity=capacite)
    
    # Remplissage des noeuds
    for i in range(dimension):
        reseau.add_node(
            node_id=i,
            demand=data['demand'][i],
            e=data['time_window'][i][0],
            l=data['time_window'][i][1],
            s=data['service_time'][i]
        )
        
    # Remplissage de la matrice des temps/distances
    distances = data['edge_weight']
    for i in reseau.V:
        for j in reseau.V:
            if i != j:
                reseau.set_travel_time(i, j, distances[i][j])
                
    return reseau

## Verificaton des fenetres de temps

In [152]:
def est_realisable(route_tuple, reseau):
    """Vérifie si une route respecte les capacités et fenêtres de temps."""
    charge_actuelle = 0
    temps_actuel = 0
    
    for k in range(len(route_tuple) - 1):
        i = route_tuple[k]
        j = route_tuple[k+1]
        
        charge_actuelle += reseau.nodes[i].q
        if charge_actuelle > reseau.Q:
            return False
            
        temps_actuel = max(temps_actuel, reseau.nodes[i].e)
        temps_actuel += reseau.nodes[i].s + reseau.travel_times.get((i, j), float('inf'))
        
        if temps_actuel > reseau.nodes[j].l:
            return False 
            
    return True

In [153]:
def solution_est_realisable(solution, reseau):
    """Vérificateur global : vérifie si TOUTES les routes de la solution sont valides."""
    for route in solution:
        if not est_realisable(route, reseau):
            return False
    return True

## Choix de la méthode de Voisinage

In [154]:
# =======================================================
# LES 3 OPÉRATEURS DE RECHERCHE LOCALE (MULTI-ROUTES)
# =======================================================

def voisinage_multi_swap(solution):
    """
    Méthode 1 : SWAP (Échange)
    Intervertit deux clients. Ils peuvent être dans la même route ou dans deux camions différents.
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r1 in range(nb_routes):
        for i in range(1, len(solution_list[r1]) - 1): # Ignore les dépôts (0)
            for r2 in range(r1, nb_routes):
                # Pour ne pas refaire les mêmes échanges en double
                start_j = i + 1 if r1 == r2 else 1
                for j in range(start_j, len(solution_list[r2]) - 1):
                    voisin = [list(r) for r in solution_list]
                    # Échange des deux clients
                    voisin[r1][i], voisin[r2][j] = voisin[r2][j], voisin[r1][i]
                    
                    yield tuple(tuple(r) for r in voisin)


def voisinage_multi_relocate(solution):
    """
    Méthode 2 : RELOCATE (Insertion)
    Retire un client de son camion actuel pour l'insérer ailleurs (même camion ou un autre).
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r1 in range(nb_routes):
        for i in range(1, len(solution_list[r1]) - 1):
            client = solution_list[r1][i]
            
            for r2 in range(nb_routes):
                for j in range(1, len(solution_list[r2])): # Peut s'insérer juste avant le dépôt final
                    if r1 == r2 and i == j: 
                        continue
                        
                    voisin = [list(r) for r in solution_list]
                    
                    # 1. On retire
                    voisin[r1].pop(i)
                    # 2. On insère
                    if r1 == r2 and j > i:
                        voisin[r2].insert(j-1, client)
                    else:
                        voisin[r2].insert(j, client)
                        
                    # On nettoie les camions devenus vides (ex: [0, 0])
                    voisin_propre = tuple(tuple(r) for r in voisin if len(r) > 2)
                    yield voisin_propre


def voisinage_multi_2opt(solution):
    """
    Méthode 3 : 2-OPT (Inversion de segment intra-route)
    Inverse totalement l'ordre de passage d'un sous-groupe de clients à l'intérieur d'un camion.
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r in range(nb_routes):
        route = solution_list[r]
        n = len(route)
        for i in range(1, n - 2):
            for j in range(i + 1, n - 1):
                voisin = [list(rt) for rt in solution_list]
                # Inversion du segment [i:j]
                voisin[r][i:j+1] = reversed(voisin[r][i:j+1])
                
                yield tuple(tuple(rt) for rt in voisin)


# =======================================================
# LE SÉLECTEUR DE MÉTHODE UNIFIÉ
# =======================================================

def voisinage_multi(solution, methode="relocate"):
    """Sélecteur global pour générer le voisinage selon la méthode choisie."""
    if methode == "swap":
        return voisinage_multi_swap(solution)
    elif methode == "relocate":
        return voisinage_multi_relocate(solution)
    elif methode == "2opt":
        return voisinage_multi_2opt(solution)
    else:
        raise ValueError(f"Méthode '{methode}' inconnue. Choisissez 'swap', 'relocate' ou '2opt'.")

## Multi Start / GARSP ? 
Explication du principe et on l implémente pour chaque algo(c est une hypothèse)

In [155]:
def random_solution_multi(reseau):
    """Génère une solution initiale valide avec PLUSIEURS camions (Glouton aléatoire)."""
    clients_restants = set(reseau.C)
    routes = []
    
    while clients_restants:
        route_actuelle = [0]
        candidats = list(clients_restants)
        random.shuffle(candidats)
        
        # On remplit le camion tant que la capacité et le temps le permettent
        for client in candidats:
            test_route = route_actuelle + [client, 0]
            # On réutilise votre fonction 'est_realisable' qui marche très bien pour une route
            if est_realisable(tuple(test_route), reseau):
                route_actuelle.append(client)
                clients_restants.remove(client)
                
        # Le camion rentre au dépôt
        route_actuelle.append(0)
        routes.append(tuple(route_actuelle))
        
    return tuple(routes) # Tuple de tuples pour être immuable (requis pour la liste Tabou)

## Heurisitcs 

### Hills Climbings

### A*

In [156]:
def hill_climbing_multi(solution_initiale, reseau):
    solution_courante = solution_initiale
    nouveau = True
    
    while nouveau:
        meilleure_solution = solution_courante
        # CORRECTION : L'appel direct (solution_courante est déjà un ensemble de routes)
        meilleur_score = reseau.calculate_objective(solution_courante)
        
        # CORRECTION : On utilise le voisinage multi-routes
        for voisin in voisinage_multi_relocate(solution_courante):
            # CORRECTION : On vérifie TOUTE la solution
            if solution_est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective(voisin)
                if score_voisin < meilleur_score:
                    meilleure_solution = voisin
                    meilleur_score = score_voisin
                    
        nouveau = (meilleure_solution != solution_courante)
        solution_courante = meilleure_solution
        
    return solution_courante

## Meta - Heurisitcs

### Tabu Search

In [157]:
from collections import deque

def recherche_tabou_multi(solution_initiale, reseau, taille_tabou, iter_max, methode_voisinage="relocate"):
    """
    Métaheuristique Tabou compatible avec plusieurs camions et les Datasets VRPLIB.
    """
    liste_tabou = deque(maxlen=taille_tabou)
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # APPEL CORRIGÉ : On passe la solution directement
    valeur_meilleure_globale = reseau.calculate_objective(solution_initiale)
    nb_iter_sans_amelioration = 0
    
    while nb_iter_sans_amelioration < iter_max:
        meilleur_voisin = None
        valeur_meilleur_voisin = float('inf')
        
        # APPEL CORRIGÉ : On utilise le sélecteur multi-voisinages
        for voisin in voisinage_multi(solution_courante, methode=methode_voisinage):
            
            # APPEL CORRIGÉ : On vérifie TOUS les camions avec solution_est_realisable
            if voisin not in liste_tabou and solution_est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective(voisin)
                
                if score_voisin < valeur_meilleur_voisin:
                    meilleur_voisin = voisin
                    valeur_meilleur_voisin = score_voisin
        
        # Si aucun voisin n'est valide (cas très rare mais possible si très contraint)
        if meilleur_voisin is None:
            break
            
        # Mise à jour des records
        if valeur_meilleur_voisin < valeur_meilleure_globale:
            meilleure_globale = meilleur_voisin
            valeur_meilleure_globale = valeur_meilleur_voisin
            nb_iter_sans_amelioration = 0
        else:
            nb_iter_sans_amelioration += 1
            
        # On avance
        solution_courante = meilleur_voisin
        liste_tabou.append(solution_courante)
        
    return meilleure_globale

### Simulated Annealing

In [158]:
import math
import random

def recuit_simule_multi(solution_initiale, reseau, temp_initiale, taux_refroidissement, iter_max, methode_voisinage="relocate"):
    """
    Métaheuristique : Recuit Simulé (Simulated Annealing) compatible avec plusieurs camions.
    Accepte des détours selon une probabilité qui baisse avec le temps.
    """
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # APPEL CORRIGÉ : On calcule le coût en passant directement le tuple de routes
    valeur_courante = reseau.calculate_objective(solution_courante)
    valeur_meilleure_globale = valeur_courante
    
    temperature = temp_initiale
    
    for _ in range(iter_max):
        # APPEL CORRIGÉ : Voisinage multi-routes et vérification globale de la validité
        voisins_valides = [
            v for v in voisinage_multi(solution_courante, methode=methode_voisinage) 
            if solution_est_realisable(v, reseau)
        ]
        
        if not voisins_valides:
            break # Sécurité : aucun voisin possible sans violer les contraintes (poids, temps)
            
        # 1. On tire UN SEUL voisin au hasard
        voisin_choisi = random.choice(voisins_valides)
        valeur_voisin = reseau.calculate_objective(voisin_choisi)
        
        # 2. Calcul de la différence de score
        # delta < 0 : le voisin est plus rapide (amélioration)
        # delta > 0 : le voisin est plus lent (dégradation)
        delta = valeur_voisin - valeur_courante
        
        # 3. Critère d'acceptation (Metropolis)
        if delta < 0:
            # C'est meilleur : on accepte d'office l'amélioration
            solution_courante = voisin_choisi
            valeur_courante = valeur_voisin
        else:
            # C'est pire : on calcule la probabilité d'accepter ce détour
            if temperature > 0.001:
                # Formule mathématique d'acceptation
                probabilite = math.exp(-delta / temperature)
                
                # Le coup de dés
                if random.random() < probabilite:
                    solution_courante = voisin_choisi
                    valeur_courante = valeur_voisin
                    
        # 4. Mise à jour du record absolu
        if valeur_courante < valeur_meilleure_globale:
            meilleure_globale = solution_courante
            valeur_meilleure_globale = valeur_courante
            
        # 5. On refroidit la température
        temperature *= taux_refroidissement
        
    return meilleure_globale

## Test et print console

In [ ]:
import vrplib

# --- 1. Chargement de l'instance et de la solution optimale ---
fichier_instance = "data/C101.txt" # Modifiez si besoin
fichier_solution = "data/C101.sol"

reseau_c101 = charger_instance_cvrptw(fichier_instance)
sol_opt = vrplib.read_solution(fichier_solution)
score_optimal = sol_opt.get("cost", sol_opt.get("Cost"))
nb_camions_optimal = len(sol_opt.get("routes", []))

print(f"--- Instance chargée : {reseau_c101.N} clients ---")

try:
    # --- 2. Point de départ aléatoire ---
    print("\nGénération de la solution initiale...")
    sol_init = random_solution_multi(reseau_c101)
    score_init = reseau_c101.calculate_objective(sol_init)
    print(f"Score Initial: {score_init:.2f} (Véhicules: {len(sol_init)})")

    # --- 3. Intensification (Hill Climbing) ---
    print("\nLancement du Hill Climbing...")
    sol_hc = hill_climbing_multi(sol_init, reseau_c101)
    score_hc = reseau_c101.calculate_objective(sol_hc)
    print(f"Score Hill Climbing: {score_hc:.2f} (Véhicules: {len(sol_hc)})")

    # --- 4. Diversification A : Recherche Tabou ---
    print("\nLancement de la Recherche Tabou (depuis le Hill Climbing)...")
    sol_tabu = recherche_tabou_multi(sol_hc, reseau_c101, taille_tabou=15, iter_max=500, methode_voisinage="relocate")
    score_tabu = reseau_c101.calculate_objective(sol_tabu)
    print(f"Score Tabou: {score_tabu:.2f} (Véhicules: {len(sol_tabu)})")

    # --- 5. Diversification B : Recuit Simulé ---
    print("\nLancement du Recuit Simulé (depuis le Hill Climbing)...")
    sol_recuit = recuit_simule_multi(sol_hc, reseau_c101, temp_initiale=100, taux_refroidissement=0.95, iter_max=500, methode_voisinage="relocate")
    score_recuit = reseau_c101.calculate_objective(sol_recuit)
    print(f"Score Recuit Simulé: {score_recuit:.2f} (Véhicules: {len(sol_recuit)})")

    # --- 6. Résultats et calcul des Gaps ---
    print("\n=== RÉSULTATS FINAUX ===")
    print(f"Solution parfaite théorique : {score_optimal} ({nb_camions_optimal} camions)")
    
    gap_hc = 100 * ((score_hc - score_optimal) / score_optimal)
    gap_tabu = 100 * ((score_tabu - score_optimal) / score_optimal)
    gap_recuit = 100 * ((score_recuit - score_optimal) / score_optimal)
    
    print(f"Gap Hill Climbing   : {gap_hc:.2f} %")
    print(f"Gap Tabou Search    : {gap_tabu:.2f} %")
    print(f"Gap Recuit Simulé   : {gap_recuit:.2f} %")

except Exception as e:
    print(f"Erreur durant l'exécution : {e}")

--- Instance chargée : 100 clients ---
Génération de la solution initiale...
Score Initial: 4298.13 (Véhicules utilisés: 35)
Lancement du Hill Climbing...
Score Hill Climbing: 988.03 (Véhicules utilisés: 13)
Lancement du Tabu Search...
Score Tabu Search: 2227.67 (Véhicules utilisés: 35)

=== RÉSULTATS ===
Score Optimal à atteindre : 827.3
Votre Gap par rapport à l'optimum : 19.43 %
La solution parfaite utilise : 10 camions.
